# 03. MLP Modeling

이 노트북은 PyTorch MLP로 `blueWins`를 예측합니다. MLP는 입력 스케일에 민감하므로 `StandardScaler`를 적용한 뒤 학습합니다.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.train_mlp import train_mlp

DATASETS = [
    (PROJECT_ROOT / "data" / "Challenger_Ranked_Games_10minute.csv", "10minute"),
    (PROJECT_ROOT / "data" / "Challenger_Ranked_Games_15minute.csv", "15minute"),
]

## 기본 MLP 구조

권장 기본 구조:

```text
Input → Linear(64) → ReLU → Dropout
      → Linear(32) → ReLU → Dropout
      → Linear(16) → ReLU → Dropout
      → Linear(1)
```

출력층에는 sigmoid를 직접 넣지 않고, 학습 시 `BCEWithLogitsLoss`를 사용합니다.

In [ ]:
results = []
for data_file, label in DATASETS:
    metrics = train_mlp(
        data_file=data_file,
        time_label=label,
        output_dir=PROJECT_ROOT / "results",
        model_dir=PROJECT_ROOT / "models",
        hidden_dims=(64, 32, 16),
        dropout=0.2,
        epochs=100,
        quick=True,   # 최종 실험에서는 False 권장
    )
    results.append(metrics)

pd.DataFrame(results)

## 저장된 결과 확인

생성되는 주요 파일:
- `results/tables/mlp_10minute_metrics.csv`
- `results/tables/mlp_15minute_metrics.csv`
- `results/tables/mlp_10minute_history.csv`
- `results/figures/mlp_10minute_loss_curve.png`
- `results/figures/mlp_10minute_confusion_matrix.png`
- `results/figures/mlp_10minute_roc_curve.png`

In [ ]:
comparison = pd.concat([
    pd.read_csv(PROJECT_ROOT / "results" / "tables" / "mlp_10minute_metrics.csv"),
    pd.read_csv(PROJECT_ROOT / "results" / "tables" / "mlp_15minute_metrics.csv"),
], ignore_index=True)
comparison[["model", "time_label", "accuracy", "precision", "recall", "f1", "roc_auc", "epochs_run"]]

In [ ]:
history10 = pd.read_csv(PROJECT_ROOT / "results" / "tables" / "mlp_10minute_history.csv")
history10.tail()

## 발표용 해석 방향

MLP는 피처 사이의 비선형 관계를 학습할 수 있지만, 이 데이터는 이미지나 텍스트가 아니라 표 형태의 tabular data입니다. 따라서 MLP가 항상 XGBoost보다 좋지는 않을 수 있습니다. 이 비교 자체가 프로젝트의 중요한 결론이 될 수 있습니다.

<!-- UPDATED_MLP_IMPROVED -->
## 최신 MLP 설정과 추가 실험

현재 `src/train_mlp.py`는 기본 MLP 구조에 더해 다음 요소를 포함합니다.

- Activation 선택 옵션: `relu`, `leaky_relu`, `gelu`, `silu`
- Optimizer: `AdamW`
- Learning rate scheduler: validation loss가 정체되면 learning rate 감소
- Early stopping: validation loss가 개선되지 않으면 학습 중단
- Threshold tuning: validation split 기준으로 decision threshold 탐색
- Prediction probability 저장: ensemble과 후속 분석에 사용

빠른 비교에서는 ReLU가 가장 안정적이었기 때문에 기본 activation은 ReLU로 유지했습니다. GELU/SiLU는 실험 옵션으로 남겨 두었습니다.

In [ ]:
# Threshold tuning이 적용된 MLP 추가 실험
mlp_threshold_results = []
for data_file, label in DATASETS:
    metrics = train_mlp(
        data_file=data_file,
        time_label=label,
        output_dir=PROJECT_ROOT / "results",
        model_dir=PROJECT_ROOT / "models",
        hidden_dims=(64, 32, 16),
        dropout=0.2,
        activation="relu",
        epochs=100,
        quick=True,
        tune_threshold=True,
        threshold_metric="accuracy",
        experiment_label="threshold",
    )
    mlp_threshold_results.append(metrics)

pd.DataFrame(mlp_threshold_results)[[
    "model", "experiment_label", "time_label", "accuracy", "precision", "recall",
    "f1", "roc_auc", "decision_threshold", "best_epoch", "best_val_loss"
]]

In [ ]:
# Activation 후보를 바꿔보고 싶을 때 사용하는 예시입니다.
# 기본값은 False로 두어 결과 파일이 불필요하게 늘어나지 않도록 했습니다.
RUN_ACTIVATION_EXAMPLE = False

if RUN_ACTIVATION_EXAMPLE:
    example_activation = "gelu"
    example_metrics = train_mlp(
        data_file=PROJECT_ROOT / "data" / "Challenger_Ranked_Games_15minute.csv",
        time_label="15minute_gelu_example",
        output_dir=PROJECT_ROOT / "results",
        model_dir=PROJECT_ROOT / "models",
        hidden_dims=(64, 32, 16),
        dropout=0.2,
        activation=example_activation,
        epochs=100,
        quick=True,
        experiment_label="activation_example",
    )
    display(pd.DataFrame([example_metrics])[[
        "model", "time_label", "activation", "accuracy", "f1", "roc_auc", "epochs_run"
    ]])
else:
    print("Activation example is skipped. Set RUN_ACTIVATION_EXAMPLE = True to run it.")

In [ ]:
# Baseline과 threshold MLP 결과 비교
mlp_compare = pd.concat([
    pd.read_csv(PROJECT_ROOT / "results" / "tables" / "mlp_10minute_metrics.csv"),
    pd.read_csv(PROJECT_ROOT / "results" / "tables" / "mlp_10minute_threshold_metrics.csv"),
    pd.read_csv(PROJECT_ROOT / "results" / "tables" / "mlp_15minute_metrics.csv"),
    pd.read_csv(PROJECT_ROOT / "results" / "tables" / "mlp_15minute_threshold_metrics.csv"),
], ignore_index=True, sort=False)

mlp_compare["experiment_label"] = mlp_compare["experiment_label"].fillna("baseline").replace("", "baseline")
mlp_compare[[
    "model", "experiment_label", "time_label", "accuracy", "precision", "recall", "f1", "roc_auc",
    "activation", "optimizer", "decision_threshold", "best_epoch"
]]

## MLP 해석 포인트

MLP는 여러 피처 사이의 비선형 관계를 학습할 수 있지만, 이 데이터는 이미지나 텍스트가 아닌 tabular data입니다. 따라서 MLP가 XGBoost보다 항상 우수할 것이라고 기대하기는 어렵습니다.

본 프로젝트에서 MLP는 XGBoost와 비슷한 ROC-AUC를 보이며 경쟁력 있는 성능을 냈지만, 단일 대표 모델로는 feature importance 해석이 가능한 XGBoost가 더 적합합니다. MLP는 XGBoost와 비교하거나 soft voting ensemble에 포함하여 보완 모델로 활용하는 것이 좋습니다.